In [1]:
import sys
import os
import pandas as pd
import numpy as np

from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, classification_report
from scipy.stats import spearmanr

In [2]:
sys.path.append(os.path.abspath(".."))

from src.config import UNIVERSE

UNIVERSE


['XLK',
 'XLC',
 'XLY',
 'XLP',
 'XLF',
 'XLV',
 'XLE',
 'XLI',
 'XLB',
 'XLRE',
 'XLU',
 'AAPL',
 'MSFT',
 'NVDA',
 'GOOGL',
 'META',
 'NFLX',
 'AMZN',
 'TSLA',
 'MCD',
 'PG',
 'KO',
 'PEP',
 'JPM',
 'BAC',
 'V',
 'JNJ',
 'PFE',
 'UNH',
 'XOM',
 'CVX',
 'COP',
 'BA',
 'CAT',
 'GE',
 'LIN',
 'SHW',
 'FCX',
 'AMT',
 'PLD',
 'EQIX',
 'NEE',
 'DUK',
 'SO']

In [3]:
import glob, os

files = glob.glob("../data/raw/D1/*.US_D1.csv")

available = {
    os.path.basename(f).split(".")[0]   # like "AAPL"
    for f in files
}

valid_universe = [t for t in UNIVERSE if t in available]
missing = [t for t in UNIVERSE if t not in available]

len(valid_universe), missing[:20]


(33,
 ['XLK',
  'XLC',
  'XLY',
  'XLP',
  'XLF',
  'XLV',
  'XLE',
  'XLI',
  'XLB',
  'XLRE',
  'XLU'])

In [4]:
dfs = []

for ticker in valid_universe:
    path = f"../data/raw/D1/{ticker}.US_D1.csv"
    df_tmp = pd.read_csv(path)
    df_tmp["ticker"] = ticker
    dfs.append(df_tmp)

df = pd.concat(dfs, ignore_index=True)

df["datetime"] = pd.to_datetime(df["datetime"])
df = df.sort_values(["ticker", "datetime"]).reset_index(drop=True)

df.shape


(200892, 7)

In [5]:
df = df[df["datetime"] >= "2010-01-04"].copy()

df.shape

(118620, 7)

In [6]:
df.head()

,datetime,open,high,low,close,volume,ticker
3019,2010-01-04,7.62,7.65,7.58,7.64,116694802,AAPL
3020,2010-01-05,7.65,7.69,7.61,7.65,136014592,AAPL
3021,2010-01-06,7.65,7.68,7.52,7.53,133300727,AAPL
3022,2010-01-07,7.56,7.57,7.46,7.52,113809059,AAPL
3023,2010-01-08,7.51,7.57,7.46,7.56,104221936,AAPL


In [7]:
df.columns

Index(['datetime', 'open', 'high', 'low', 'close', 'volume', 'ticker'], dtype='str')

In [8]:
df.duplicated(subset=["ticker", "datetime"]).sum()

np.int64(0)

In [9]:
df.isna().sum()

datetime    0
open        0
high        0
low         0
close       0
volume      0
ticker      0
dtype: int64

In [10]:
df.groupby("ticker")["datetime"].is_monotonic_increasing.all()

np.True_

In [11]:
df.groupby("ticker").size().describe()

count      33.000000
mean     3594.545455
std       106.534939
min      3021.000000
25%      3619.000000
50%      3620.000000
75%      3620.000000
max      3620.000000
dtype: float64

In [12]:
from src.features import add_features

df = add_features(df)

df.shape

(114462, 18)

In [13]:
from src.labels_long import add_long_score

df = add_long_score(df)

df.shape

(112383, 22)

In [14]:
df["ret_long_norm"].describe()

count    112383.000000
mean          1.737777
std          10.505143
min         -75.973971
25%          -2.379676
50%           1.635526
75%           5.713470
max         425.753798
Name: ret_long_norm, dtype: float64

In [15]:
dates = pd.Index(df["datetime"].unique()).sort_values()

min_train_days = 756   # ~3 years
test_days = 126        # ~6 months
step_days = 63         # ~3 months

splits = []
for i in range(min_train_days, len(dates) - test_days, step_days):
    train_end = dates[i-1]
    test_start = dates[i]
    test_end = dates[i + test_days - 1]
    splits.append((train_end, test_start, test_end))

len(splits), splits[0], splits[-1]


(41,
 (Timestamp('2013-07-05 00:00:00'),
  Timestamp('2013-07-08 00:00:00'),
  Timestamp('2014-01-03 00:00:00')),
 (Timestamp('2023-05-25 00:00:00'),
  Timestamp('2023-05-26 00:00:00'),
  Timestamp('2023-11-21 00:00:00')))

In [16]:
train_end, test_start, test_end = splits[0]
train = df[df["datetime"] <= train_end]
test  = df[(df["datetime"] >= test_start) & (df["datetime"] <= test_end)]

train.shape, test.shape

((24226, 22), (4158, 22))

In [17]:
def make_cs_label(df, col):
    df = df.copy()
    df["rank_pct"] = df.groupby("datetime")[col].rank(pct=True)

    df["long_label"] = "Neutral"

    # swap Up/Down
    df.loc[df["rank_pct"] >= 0.7, "long_label"] = "Down"
    df.loc[df["rank_pct"] <= 0.3, "long_label"] = "Up"

    return df


train = make_cs_label(train, "ret_long_norm")
test  = make_cs_label(test,  "ret_long_norm")


In [18]:
train["long_label"].value_counts()
test["long_label"].value_counts()

long_label
Neutral    1764
Down       1260
Up         1134
Name: count, dtype: int64

In [19]:
train_bin = train[train["long_label"] != "Neutral"].copy()
test_bin  = test[test["long_label"] != "Neutral"].copy()

y_train = (train_bin["long_label"] == "Up").astype(int)
y_test  = (test_bin["long_label"] == "Up").astype(int)

print(len(train_bin), len(test_bin))
print(y_train.mean(), y_test.mean())

14364 2394
0.47368421052631576 0.47368421052631576


In [20]:
feature_cols = [
    "ret_1",
    "ret_5",
    "ret_21",
    "mom_63",
    "mom_126",
    "vol_21",
    "vol_63",
    "ma_ratio_21_63",
    "drawdown_63"
]

In [21]:
def precision_at_topk(y_true, y_score, topk_frac=0.10):
    n = len(y_true)
    k = max(1, int(n * topk_frac))
    idx = np.argsort(y_score)[::-1][:k]
    return float(y_true.iloc[idx].mean())

def ic_spearman(y_score, fwd_ret):
    corr, _ = spearmanr(y_score, fwd_ret, nan_policy="omit")
    return float(corr) if corr == corr else np.nan


rows = []

for k in range(5):

    train_end, test_start, test_end = splits[k]

    train = df[df["datetime"] <= train_end].copy()
    test  = df[(df["datetime"] >= test_start) & (df["datetime"] <= test_end)].copy()

    # Cross-sectional labeling
    train["rank_pct"] = train.groupby("datetime")["ret_long_norm"].rank(pct=True)
    test["rank_pct"]  = test.groupby("datetime")["ret_long_norm"].rank(pct=True)

    train["long_label"] = np.where(
        train["rank_pct"] >= 0.7, "Up",
        np.where(train["rank_pct"] <= 0.3, "Down", "Neutral")
    )

    test["long_label"] = np.where(
        test["rank_pct"] >= 0.7, "Up",
        np.where(test["rank_pct"] <= 0.3, "Down", "Neutral")
    )

    # Drop Neutral
    train_bin = train[train["long_label"] != "Neutral"].copy()
    test_bin  = test[test["long_label"] != "Neutral"].copy()

    y_train = (train_bin["long_label"] == "Up").astype(int)
    y_test  = (test_bin["long_label"] == "Up").astype(int)

    X_train = train_bin[feature_cols]
    X_test  = test_bin[feature_cols]

    model = LGBMClassifier(
        n_estimators=800,
        learning_rate=0.03,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    proba_up = model.predict_proba(X_test)[:, 1]

    # IMPORTANT: flipped score
    score_up = 1 - proba_up

    auc = roc_auc_score(y_test, score_up)

    p10 = precision_at_topk(
        y_test.reset_index(drop=True),
        pd.Series(score_up),
        0.10
    )

    ic = ic_spearman(
        score_up,
        test_bin["ret_long_norm"].values
    )

    rows.append({
        "fold": k,
        "auc": auc,
        "p10": p10,
        "ic": ic
    })

    print(f"{k} | AUC: {auc:.4f} | P@10: {p10:.4f} | IC: {ic:.4f}")


res = pd.DataFrame(rows)

print("\n====================")
print("RESULT TABLE")
print(res)

print("\nMEANS")
print(res[["auc","p10","ic"]].mean())

[LightGBM] [Info] Number of positive: 7560, number of negative: 6804
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002482 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2295
[LightGBM] [Info] Number of data points in the train set: 14364, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.526316 -> initscore=0.105361
[LightGBM] [Info] Start training from score 0.105361
0 | AUC: 0.5379 | P@10: 0.5816 | IC: 0.0401
[LightGBM] [Info] Number of positive: 8190, number of negative: 7371
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000963 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2295
[LightGBM] [Info] Number of data points in the train set: 15561, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pav

In [23]:
import os
os.makedirs("models", exist_ok=True)

In [24]:
import joblib
joblib.dump(model, "models/long_model.pkl")

['models/long_model.pkl']